# 09: How many measurements are enough?

**Level:** Beginner to intermediate  
**Before you start:** Notebook 01.  
**Resources:** CPU unless an optional remote step is enabled.

Why do two executions of the same circuit give different counts?

Run each cell in order. All core calculations are written in this notebook.

## 1. Start with a probability we know

For RY(theta)|0⟩, P(1) = sin²(theta/2). We can compare sampled counts against this analytic reference.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import flagquantum as fq

angle = 1.1
q = fq.Circuit(1).ry(0, angle)
p1 = math.sin(angle / 2) ** 2
print("Exact P(1):", p1)


## 2. Ask FlagQuantum for counts

A seed makes this local sampling experiment reproducible. It does not turn hardware measurements into deterministic results.
`result.counts` contains one dictionary per circuit in the batch. We use `[0]` because this experiment has one circuit.


In [ ]:
shots = 128
sampled = fq.run(
    q, outputs=fq.counts(), shots=shots, options=fq.ExecutionOptions(seed=42)
)
counts = sampled.counts[0]
print(counts)
assert sum(counts.values()) == shots
print("Estimated P(1):", counts.get("1", 0) / shots)


## 3. Repeat with different shot budgets

Each point below is a separate seeded experiment. More shots should reduce typical variation, though any single experiment can be lucky or unlucky.

In [ ]:
budgets = [32, 128, 512, 2048]
estimates = []
for n in budgets:
    values = []
    for seed in range(12):
        r = fq.run(
            q, outputs=fq.counts(), shots=n, options=fq.ExecutionOptions(seed=seed)
        )
        values.append(r.counts[0].get("1", 0) / n)
    estimates.append(values)
plt.boxplot(estimates)
plt.xticks(range(1, len(budgets) + 1), [str(n) for n in budgets])
plt.axhline(p1, color="black", linestyle="--", label="Exact")
plt.xlabel("Shots per experiment")
plt.ylabel("Estimated P(1)")
plt.legend()
plt.show()
print(
    "Theoretical standard deviations:", [math.sqrt(p1 * (1 - p1) / n) for n in budgets]
)


## Make it yours

Predict how many times more shots you need to halve the standard deviation. Try an angle close to zero. Explain why a narrow spread around the wrong answer could still indicate device bias.